# Data & Modèle — Classifieur d'espèces de plantes

Partie : chargement du dataset, construction du modèle (MobileNetV2, transfer learning), entraînement, sauvegarde.

Dataset : Oxford 102 Flowers (`tensorflow_datasets`, `oxford_flowers102`).

In [ ]:
!pip install -q tensorflow tensorflow-datasets matplotlib

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

## 1. Chargement et inspection du dataset

In [ ]:
(ds_train, ds_val, ds_test), ds_info = tfds.load(
    "oxford_flowers102",
    split=["train", "validation", "test"],
    as_supervised=True,
    with_info=True,
)

NUM_CLASSES = ds_info.features["label"].num_classes
CLASS_NAMES = ds_info.features["label"].names
print(f"Nombre de classes : {NUM_CLASSES}")
print(f"Train : {ds_info.splits['train'].num_examples} exemples")
print(f"Validation : {ds_info.splits['validation'].num_examples} exemples")
print(f"Test : {ds_info.splits['test'].num_examples} exemples")

In [ ]:
# Inspection des shapes et exemples d'images
for image, label in ds_train.take(1):
    print("Shape image :", image.shape, "dtype :", image.dtype)
    print("Label :", label.numpy(), CLASS_NAMES[label.numpy()])

plt.figure(figsize=(10, 10))
for i, (image, label) in enumerate(ds_train.take(9)):
    plt.subplot(3, 3, i + 1)
    plt.imshow(image)
    plt.title(CLASS_NAMES[label.numpy()])
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Distribution des classes (train) - vérifier l'équilibre avant de décider le split
labels = [int(label.numpy()) for _, label in ds_train]
counts = np.bincount(labels, minlength=NUM_CLASSES)

plt.figure(figsize=(14, 4))
plt.bar(range(NUM_CLASSES), counts)
plt.xlabel("Classe")
plt.ylabel("Nombre d'exemples (train)")
plt.title("Distribution des classes")
plt.show()

print(f"Min par classe : {counts.min()}, Max par classe : {counts.max()}")

## 2. Préparation des pipelines (resize, normalisation, batch)

In [ ]:
def preprocess(image, label):
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    label = tf.one_hot(label, NUM_CLASSES)
    return image, label


train_ds = (
    ds_train.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds = ds_val.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = ds_test.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

## 3. Construction du modèle (MobileNetV2, couches gelées + tête de classification)

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMAGE_SIZE + (3,),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False  # couches gelées : on ne réentraîne que la tête

inputs = tf.keras.Input(shape=IMAGE_SIZE + (3,))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs * 255.0)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)
model.summary()

## 4. Compilation et entraînement

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
)

In [ ]:
# Courbes loss / accuracy train vs val — pour diagnostiquer over/underfitting
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.title("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="val")
plt.axhline(y=1 / NUM_CLASSES, color="red", linestyle="--", label="baseline aléatoire")
plt.title("Accuracy")
plt.legend()

plt.show()

## 5. Évaluation sur le test set et sauvegarde

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)
print(f"Test accuracy : {test_accuracy:.2%} (baseline aléatoire : {1 / NUM_CLASSES:.2%})")

In [ ]:
model.save("../models/model.keras")
print("Modèle sauvegardé dans models/model.keras")
print("Si le fichier dépasse 100 Mo : le mettre sur Google Drive et ajouter le lien dans le README.")